This notebook is mostly a sanity check that the isochrone works reasonably. 

In [ ]:
from astropy.coordinates import SkyCoord

In [ ]:
import numpy as np

In [ ]:
from read_iso import ISOCMD

In [ ]:
from astropy.table import Table

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

isos_fe_h = ["m1.00", "m1.25", "m1.50", "m1.75", "m2.00", "m2.50", "m3.00"]

isonames = [f"../MIST/MIST_v1.2_vvcrit0.4_SDSSugriz/MIST_v1.2_feh_{iso_fe_h}_afe_p0.0_vvcrit0.4_SDSSugriz.iso.cmd" for iso_fe_h in isos_fe_h]
isochrones = {fe_h: ISOCMD(name) for fe_h, name in zip(isos_fe_h, isonames)}



# Setup

The below sql creates the sample


```
SELECT u,g,r,i,z,ra,dec, flags_r, err_g, err_r, err_i, err_z
FROM Star
WHERE
ra BETWEEN 211.36371-0.5 and 211.36371+0.5 AND dec BETWEEN 28.53444-0.5 and 28.53444+0.5
AND ((flags_r & 0x10000000) != 0)
-- detected in BINNED1
AND ((flags_r & 0x8100000c00a4) = 0)
-- not EDGE, NOPROFILE, PEAKCENTER, NOTCHECKED, PSF_FLUX_INTERP,
-- SATURATED, or BAD_COUNTS_ERROR
AND (((flags_r & 0x400000000000) = 0) or (psfmagerr_r <= 0.2))
-- not DEBLEND_NOPEAK or small PSF error
-- (substitute psfmagerr in other band as appropriate)
AND (((flags_r & 0x100000000000) = 0) or (flags_r & 0x1000) = 0)
-- not INTERP_CENTER or not COSMIC_RAY
``` 

In [ ]:
ra0 = 211.36371
dec0 = 28.53444
coord0 = SkyCoord(ra0, dec0, unit="deg")

In [ ]:
df_all = Table.read("ngc_5466.csv")
r2 = (df_all["ra"] - ra0)**2 + (df_all["dec"] - dec0)**2


In [ ]:
df = df_all[r2 < 0.15**2]

In [ ]:
plt.hist(r2)

In [ ]:
plt.hist(df["g"], np.linspace(0, 28, 100))

In [ ]:
from filter_utils import plot_iso
from ana_utils import get_extinction, fit_err
import filter_utils

In [ ]:
iso = isochrones["m1.50"][10.1]
iso = iso[iso["phase"] < 4]

In [ ]:
dm = 16.02
A = 0.05

In [ ]:
from dustmaps.sfd import SFDQuery

In [ ]:
Ebv = SFDQuery()(coord0)
A_g, A_r, A_i = get_extinction(3.1 * Ebv)

In [ ]:
params = filter_utils.filter_params(A_V =3.1*Ebv, dm=dm, 
                                    iso_fe_h="m2.00",
                                    iso_log_age=10.2,
                                    iso_width=0.05
                                   )

In [ ]:
gr_err = fit_err(df["g"], df["err_g"] + df["err_r"])
ri_err = fit_err(df["r"], df["err_r"] + df["err_i"])

In [ ]:
plt.scatter(df["r"] - df["i"], df["r"], s=0.5, lw=0, zorder=-5)
plt.ylim(23, 14)
plt.xlim(-1, 3)

# plt.plot(iso["SDSS_r"] - iso["SDSS_i"] + A_r - A_i, iso["SDSS_r"] + dm + A_r, color="k")

filter_utils.plot_iso_ri(params, ri_err)
plt.xlabel("r - i")
plt.ylabel("r")
plt.title("NGC 5466")

In [ ]:
plt.scatter(df["g"] - df["r"], df["g"], s=0.5, lw=0, zorder=-5)
plt.ylim(24, 14)
plt.xlim(-1, 3)
# plt.plot(iso["SDSS_g"] - iso["SDSS_r"] + A_g - A_r, iso["SDSS_g"] + A_g + dm, color="k")
filter_utils.plot_iso_gr(params, ri_err)

plt.xlabel("g - r")
plt.ylabel("g")
plt.title("NGC 5466")

In [ ]:
plt.scatter(df["ra"], df["dec"], s=1)


# Varying the isochrone parameters

In [ ]:
from copy import copy

In [ ]:
import arya

In [ ]:
def plot_ri_iso(dm=dm, A_V=0.1, iso_fe_h="m2.00", iso_log_age=10.1, **kwargs):
    iso = isochrones[iso_fe_h][iso_log_age]
    iso = iso[iso["phase"] < 4]

    A_g, A_r, A_i = get_extinction(A_V)
    plt.plot(iso["SDSS_r"] - iso["SDSS_i"] + A_r - A_i, iso["SDSS_r"] + dm + A_r, zorder=-5, **kwargs)

    plt.xlabel("r - i")
    plt.ylabel("r")
    plt.gca().invert_yaxis()

In [ ]:
def plot_gr_iso(dm=dm, A_V=0.1, iso_fe_h="m2.00", iso_log_age=10.1, **kwargs):
    iso = isochrones[iso_fe_h][iso_log_age]
    iso = iso[iso["phase"] < 4]

    A_g, A_r, A_i = get_extinction(A_V)
    plt.plot(iso["SDSS_g"] - iso["SDSS_r"] + A_g - A_r, iso["SDSS_g"] + dm + A_g, zorder=-5, **kwargs)

    plt.xlabel("g - r")
    plt.ylabel("g")
    plt.gca().invert_yaxis()

In [ ]:
hm = arya.HueMap((0, 0.5))

fig, axs = plt.subplots(1, 2)

plt.sca(axs[0])
for A_V in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    plot_gr_iso(A_V=A_V, color=hm(A_V), label="A_V = " + str(A_V))

filter_utils.plot_iso_gr(params, gr_err)
plt.xlim(0, 1.5)
plt.ylim(25, 12)


plt.sca(axs[1])
for A_V in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    plot_ri_iso(A_V=A_V, color=hm(A_V), label="A_V = " + str(A_V))
plt.ylim(25, 12)
filter_utils.plot_iso_ri(params, ri_err)
plt.xlim(-0.5, 1)


arya.Legend(-1)
plt.ylim(25, 12)

In [ ]:
hm = arya.HueMap((1e9, 2e10))

fig, axs = plt.subplots(1, 2)
plt.sca(axs[0])

for age in [9.5, 9.7, 9.9, 10, 10.1, 10.2, 10.3]:
    plot_gr_iso(iso_log_age=age, color=hm(10**age), label=str(round(10**age / 1e9)) + " Gyr")
plt.ylim(25, 12)
filter_utils.plot_iso_gr(params, gr_err)
plt.xlim(0, 1.5)


plt.sca(axs[1])
for age in [9.5, 9.7, 9.9, 10, 10.1, 10.2, 10.3]:
    plot_ri_iso(iso_log_age=age, color=hm(10**age), label=str(round(10**age / 1e9)) + " Gyr")
filter_utils.plot_iso_ri(params, ri_err)
plt.xlim(-0.5, 1)

arya.Legend(-1)
plt.ylim(25, 12)

In [ ]:
hm = arya.HueMap((-3, -1))
fig, axs = plt.subplots(1, 2)
plt.sca(axs[0])


for fe_h in isos_fe_h:
    x = -float(fe_h[1:])
    print(x)
    plot_gr_iso(iso_fe_h=fe_h, color=hm(x), label=fe_h)
plt.ylim(25, 12)
filter_utils.plot_iso_gr(params, gr_err)
plt.xlim(0, 1.5)


plt.sca(axs[1])

for fe_h in isos_fe_h:
    x = -float(fe_h[1:])
    print(x)
    plot_ri_iso(iso_fe_h=fe_h, color=hm(x), label=fe_h)
filter_utils.plot_iso_ri(params, ri_err)
plt.xlim(-0.5, 1)

arya.Legend(-1)
plt.ylim(25, 12)